# 20 — H1: Do 311 quality-of-life complaints lead felony reports?

**Hypothesis:** QoL-311 month *t* correlates positively with major-felony month
*t+k* for some k in [3, 12], at the precinct level, with r at lag k materially
above r at lag 0. **Non-obvious check:** survives within mid-income tracts.


In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from nyc_eda import DUCKDB_PATH, CUTS_OUT
from nyc_eda.storage import open_db
from nyc_eda import cuts
from nyc_eda.stats import lagged_correlation, permutation_test_lag

con = open_db(DUCKDB_PATH)


## C16 — QoL-311 × major-felony, monthly per precinct


In [ ]:
panel = cuts.c16_qol311_x_majorfelony(con)
panel.to_csv(CUTS_OUT / 'C16_qol_x_felony.csv', index=False)
print(f'panel: {len(panel):,} rows over {panel["precinct"].nunique()} precincts, '
      f'{panel["month"].nunique()} months')
panel.head()


## C17 — Lag scan, per precinct


In [ ]:
lag_results = cuts.c17_lag_scan_per_precinct(panel, max_lag=12)
lag_results.to_csv(CUTS_OUT / 'C17_lag_scan.csv', index=False)
median_by_lag = lag_results.groupby('lag')['r'].median().reset_index()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(median_by_lag['lag'], median_by_lag['r'], marker='o')
ax.axhline(0, color='gray', lw=0.5); ax.set_xlabel('lag (months)')
ax.set_ylabel('median Pearson r across precincts')
ax.set_title('C17 — QoL-311 → major-felony lag scan')
plt.tight_layout(); plt.savefig(CUTS_OUT / 'C17_lag_scan.png', dpi=120); plt.show()
median_by_lag


## City-level lag scan and permutation test


In [ ]:
city = panel.groupby('month').agg(qol_311=('qol_311','sum'), major_felony=('major_felony','sum')).reset_index()
lc = lagged_correlation(city['qol_311'], city['major_felony'], max_lag=12)
print('city-level lag scan:'); print(lc)
best_lag = int(lc.iloc[lc['r'].idxmax()]['lag'])
perm = permutation_test_lag(city['qol_311'], city['major_felony'], lag=best_lag, n_perm=2000)
print(f'best lag = {best_lag} months, perm test: {perm}')


## Verdict


In [ ]:
best_r = lc['r'].max()
lag_at_best = int(lc.iloc[lc['r'].idxmax()]['lag'])
lag0_r     = float(lc[lc['lag']==0]['r'].iloc[0]) if (lc['lag']==0).any() else float('nan')
verdict = 'partial'
if best_r >= 0.35 and (best_r - lag0_r) > 0.05 and perm['p_perm'] < 0.05:
    verdict = 'yes'
elif best_r < 0.20:
    verdict = 'no'
print(f'SIGNAL: {verdict}')
print(f'  best_r={best_r:.3f} at lag={lag_at_best}, lag0_r={lag0_r:.3f}, perm_p={perm["p_perm"]:.3f}')


## Serendipity
